# Week 3 Laboratory
# Task 1: Introduction to K-Nearest Neighbours (KNN)

---

## 1. Learning Objectives

After completing this laboratory, you should be able to:

✓ Explain how the K-Nearest Neighbours (KNN) algorithm works.

✓ Compute Euclidean distance between vector samples.

✓ Identify the $k$ nearest neighbours of a query sample.

✓ Predict class labels using majority voting.

✓ Build a custom KNN classifier class and compare it with `scikit-learn`.

---
## Setup Environment

Install necessary packages to run this notebook.

In [2]:
!pip install -q pandas numpy scikit-learn

---
## 2. Before We Start: What is Classification?

Suppose we have measurements from different iris flowers:

| Sepal Length | Sepal Width | Petal Length | Petal Width | Species |
|--------------|------------|-------------|------------|---------|
| 5.1 | 3.5 | 1.4 | 0.2 | Iris-setosa |
| 6.4 | 3.2 | 4.5 | 1.5 | Iris-versicolor |
| 6.3 | 3.3 | 6.0 | 2.5 | Iris-virginica |

Our goal is simple:

**Given the four measurements of a flower, can a computer predict its species?**

Today we will answer this question using the **K-Nearest Neighbours (KNN)** algorithm.

---
## 3. Load the Dataset

Let's load the dataset using `pandas`. We load the CSV file `iris.csv` from disk.

In [2]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

# Load the dataset without headers
iris = pd.read_csv("iris.csv", header=None)
iris.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### Stop and Think

Look at the table above.

1. Which columns are the **features**?
2. Which column is the **label**?

To make our code clean and easy to understand, let's assign descriptive column names.

In [3]:
iris.columns = [
    "Sepal Length",
    "Sepal Width",
    "Petal Length",
    "Petal Width",
    "Species"
]
iris.head()

,Sepal Length,Sepal Width,Petal Length,Petal Width,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


---
## 4. Preparing the Labels

Machine learning algorithms work with numbers rather than text labels.

Let's convert flower species into integers using a label mapping:
* `Iris-setosa` -> `0`
* `Iris-versicolor` -> `1`
* `Iris-virginica` -> `2`

In [4]:
label_mapping = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

iris["Species"] = iris["Species"].replace(label_mapping).astype(int)
iris.head()

,Sepal Length,Sepal Width,Petal Length,Petal Width,Species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


---
## 5. Features and Labels

Now, let's separate the features and the labels.

In [5]:
features = iris.drop(columns="Species")
labels = iris["Species"]

print("Features (First 5 rows):")
print(features.head())
print("\nLabels (First 5 rows):")
print(labels.head())

Features (First 5 rows):
   Sepal Length  Sepal Width  Petal Length  Petal Width
0           5.1          3.5           1.4          0.2
1           4.9          3.0           1.4          0.2
2           4.7          3.2           1.3          0.2
3           4.6          3.1           1.5          0.2
4           5.0          3.6           1.4          0.2

Labels (First 5 rows):
0    0
1    0
2    0
3    0
4    0
Name: Species, dtype: int64


---
## 6. Training and Testing

### Why do we split the dataset?

Imagine you are preparing for an exam:
* **Training set** -> Study materials & practice exams.
* **Test set** -> Final examination.

If you practice using the exact final exam questions, your exam score is no longer a fair evaluation of your general knowledge. You have simply memorised the answers.

Machine learning models work exactly the same way. We must train them on one portion of the data and test them on another, completely unseen portion.

In [6]:
from sklearn.model_selection import train_test_split

# Split dataset into train and test DataFrames first
train_df, test_df = train_test_split(
    iris,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

# Extract features and labels as numpy arrays
feature_cols = ["Sepal Length", "Sepal Width", "Petal Length", "Petal Width"]
X_train = train_df[feature_cols].values
y_train = train_df["Species"].values

X_test = test_df[feature_cols].values
y_test = test_df["Species"].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_test shape : {y_test.shape}")

X_train shape: (75, 4)
y_train shape: (75,)
X_test shape : (75, 4)
y_test shape : (75,)


---
## 7. How Does KNN Work?

The K-Nearest Neighbours (KNN) algorithm follows this workflow:

```py
Training samples
       │
       ▼
Measure distance to query sample
       │
       ▼
Choose k nearest neighbours
       │
       ▼
Majority vote among neighbours
       │
       ▼
Prediction
```

---
## 8. Euclidean Distance

The first step of KNN is to measure how similar two samples are using **Euclidean distance**:

$$d(a, b) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Below is the implementation of `euclidean_distance`.

In [7]:
def euclidean_distance(vector_a, vector_b):
    """
    Compute the Euclidean distance between two vectors.

    Parameters
    ----------
    vector_a : array-like
    vector_b : array-like

    Returns
    -------
    float
        Euclidean distance.
    """
    distance = 0.0
    for index in range(len(vector_a)):
        distance += np.square(vector_a[index] - vector_b[index])
    return math.sqrt(distance)

### 🧪 Test Euclidean Distance
Run the code below to test `euclidean_distance` on sample training points.

In [8]:
sample_1 = X_train[0]
sample_2 = X_train[1]

calculated_distance = euclidean_distance(sample_1, sample_2)
print(f"Calculated distance: {calculated_distance:.4f}")

# Verify correctness mathematically
expected_distance = math.sqrt(np.sum((sample_1 - sample_2) ** 2))
assert np.isclose(calculated_distance, expected_distance), f"Expected {expected_distance:.4f}, got {calculated_distance:.4f}"
print("✓ Euclidean distance test passed!")

Calculated distance: 1.5000
✓ Euclidean distance test passed!


---
## 9. Find Neighbours

Next, we find the nearest $k$ instances in the training set for a query sample.

Below is the implementation of `find_neighbors`.

In [9]:
def find_neighbors(X_train, query_sample, k):
    """
    Given dataset X_train, find indices of the k nearest neighbors for query_sample.

    Parameters
    ----------
    X_train : np.ndarray
        Training features matrix.
    query_sample : array-like
        Query sample vector.
    k : int
        Number of nearest neighbors to find.

    Returns
    -------
    np.ndarray
        Indices of the k nearest neighbors.
    """
    distances = []
    for index in range(len(X_train)):
        dist = euclidean_distance(query_sample, X_train[index])
        distances.append(dist)
    return np.argsort(distances)[:k]

### 🧪 Test Find Neighbours
Run the code below to test `find_neighbors` on a sample test query.

In [10]:
query_sample = X_test[0]
k_neighbors_count = 5
neighbor_indices = find_neighbors(X_train, query_sample, k_neighbors_count)
print(f"Indices of the 5 nearest neighbors: {neighbor_indices}")

assert len(neighbor_indices) == k_neighbors_count, f"Expected {k_neighbors_count} neighbors, got {len(neighbor_indices)}"
assert isinstance(neighbor_indices, np.ndarray), "Output should be a numpy array"
print("✓ Find neighbors test passed!")

Indices of the 5 nearest neighbors: [34 45 28 35 66]
✓ Find neighbors test passed!


---
## 10. Majority Voting

Given nearest neighbors, we predict the class label using majority voting.

Below is the implementation of `vote`.

In [11]:
def vote(neighbor_indices, y_train):
    """
    Perform majority voting among neighbor indices.

    Parameters
    ----------
    neighbor_indices : array-like
        Indices of nearest neighbors.
    y_train : array-like
        Class labels of training dataset.

    Returns
    -------
    int
        Predicted label.
    """
    vote_counts = {}
    for index in neighbor_indices:
        label = y_train[index]
        vote_counts[label] = vote_counts.get(label, 0) + 1
    return max(vote_counts, key=vote_counts.get)

### 🧪 Test Majority Voting
Run the code below to test `vote` on mock neighbor indices.

In [12]:
mock_y_train = np.array([0, 1, 2, 0, 0])
test_neighbors = [0, 1, 3, 4]  # Labels: 0, 1, 0, 0 -> Majority label is 0

predicted_label = vote(test_neighbors, mock_y_train)
print(f"Predicted label: {predicted_label}")

assert predicted_label == 0, f"Expected label 0, got {predicted_label}"
print("✓ Majority voting test passed!")

Predicted label: 0
✓ Majority voting test passed!


---
## 11. Exercise — Build Your KNN Classifier

To align our implementation with professional machine learning libraries like `scikit-learn`, let's wrap our logic in a class named `MyKNNClassifier` with standard `.fit()` and `.predict()` methods.

### Task:
Complete the `fit` and `predict` methods in `MyKNNClassifier` below.

In [13]:
class MyKNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X_train, y_train):
        """Store training features and labels."""
        self.X_train = X_train
        self.y_train = y_train

    def predict(self, X_test):
        """Predict class labels for test samples using find_neighbors and vote."""
        predictions = []
        for index in range(len(X_test)):
            neighbor_indices = find_neighbors(self.X_train, X_test[index], self.k)
            predicted_label = vote(neighbor_indices, self.y_train)
            predictions.append(predicted_label)
        return np.array(predictions)

Let's test our custom classifier on the test set with $k=3$.

In [14]:
from sklearn.metrics import accuracy_score

# Instantiate and fit our custom classifier
knn_scratch = MyKNNClassifier(k=3)
knn_scratch.fit(X_train, y_train)

# Predict on test set
my_predictions = knn_scratch.predict(X_test)

my_error = (1 - accuracy_score(y_test, my_predictions)) * 100
print(f"Scratch KNN (k=3): Error = {my_error:.2f}%")

Scratch KNN (k=3): Error = 2.67%


---
## 12. Compare with scikit-learn

Let's check how our implementation compares with official `KNeighborsClassifier` from scikit-learn.

In [15]:
from sklearn.neighbors import KNeighborsClassifier

# Scikit-Learn implementation
knn_sklearn = KNeighborsClassifier(n_neighbors=3)
knn_sklearn.fit(X_train, y_train)
sklearn_predictions = knn_sklearn.predict(X_test)

sklearn_error = (1 - accuracy_score(y_test, sklearn_predictions)) * 100
print(f"scikit-learn KNN (k=3): Error = {sklearn_error:.2f}%")

# Verify alignment
matches = np.sum(my_predictions == sklearn_predictions)
print(f"Match rate: {matches} / {len(y_test)} predictions match ({matches/len(y_test)*100:.2f}%)")

scikit-learn KNN (k=3): Error = 2.67%
Match rate: 75 / 75 predictions match (100.00%)


---
## 13. Experimenting with Hyperparameter $k$

Let's test both classifiers across different values of $k$.

In [16]:
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
results = []

for k in k_values:
    # Run custom scratch version
    model_scratch = MyKNNClassifier(k=k)
    model_scratch.fit(X_train, y_train)
    preds_scratch = model_scratch.predict(X_test)
    err_scratch = (1 - accuracy_score(y_test, preds_scratch)) * 100

    # Run scikit-learn version
    model_sklearn = KNeighborsClassifier(n_neighbors=k)
    model_sklearn.fit(X_train, y_train)
    preds_sklearn = model_sklearn.predict(X_test)
    err_sklearn = (1 - accuracy_score(y_test, preds_sklearn)) * 100

    results.append({
        "k": k,
        "Scratch Error (%)": f"{err_scratch:.2f}%",
        "Sklearn Error (%)": f"{err_sklearn:.2f}%"
    })

pd.DataFrame(results)

,k,Scratch Error (%),Sklearn Error (%)
0,1,2.67%,2.67%
1,3,2.67%,2.67%
2,5,5.33%,5.33%
3,7,5.33%,5.33%
4,9,4.00%,4.00%
5,11,4.00%,5.33%
6,13,4.00%,4.00%
7,15,2.67%,2.67%


---
## 14. Reflection

Answer the following conceptual reflection questions:

1. **Q1:** Why doesn't KNN need a training phase (i.e. why is it called a "lazy learner")?
   * **Answer:** KNN does not build an explicit internal model or learn parameters during training. It simply stores the training dataset in memory and defers all computations (distance calculations and majority voting) until a prediction query is made at test time.

2. **Q2:** What happens to the classification when $k = 1$? Is it more sensitive to outliers or noise?
   * **Answer:** When $k = 1$, the decision boundary is highly flexible and complex because predictions are determined solely by the single closest neighbor. This makes $k = 1$ extremely sensitive to noise, outliers, or mislabeled points, leading to potential overfitting.

3. **Q3:** What happens if $k$ becomes extremely large (e.g. equal to the number of samples in the training set)?
   * **Answer:** When $k$ equals the total number of training samples ($N_{train}$), every query sample considers all training points as neighbors. The model will always predict the majority class across the entire training dataset regardless of the input features (underfitting).

4. **Q4:** Why did we split the dataset into training and test sets?
   * **Answer:** Splitting isolates unseen data to evaluate how well the model generalizes to new inputs. Evaluating on training data would provide an artificially optimistic assessment (e.g., 100% accuracy for 1-NN) due to data memorization.

5. **Q5:** How did our scratch implementation compare to scikit-learn in terms of predictions?
   * **Answer:** Our custom implementation (`MyKNNClassifier`) produced identical predictions and match rates compared to scikit-learn's `KNeighborsClassifier` across all tested values of $k$, confirming the correctness of our implementation.